# 064 — Tokenización y representación del lenguaje

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

**Tokenización:** por palabras (vocabulario enorme + `<UNK>`) y por caracteres (secuencias
larguísimas) son extremos fallidos; las **subpalabras** son el punto medio. **BPE** parte
de caracteres y fusiona iterativamente el par adyacente más frecuente hasta el tamaño de
vocabulario deseado; el texto nuevo se tokeniza aplicando las fusiones en orden. GPT usa
BPE sobre **bytes** (nunca hay `<UNK>`); **WordPiece** (BERT) marca fragmentos con `##`;
**SentencePiece** trata el espacio como símbolo (`▁`).

**Representación:** one-hot (sin similitud) → bolsa de palabras (conteos, sin orden) →
TF-IDF (conteos ponderados por rareza) → **embeddings densos**: tabla `|V|×d` aprendida
donde tokens de contextos similares acaban cerca.

**Consecuencias:** costo y ventana de los LLM se miden en tokens (~1.4–1.8 por palabra en
español); los idiomas subrepresentados se fragmentan más y pagan más.


### 🧮 BPE de referencia (para los ejercicios)

```text
Corpus: low ×5, lower ×2, newest ×6, widest ×3   (con marcador final _)
Par más frecuente: (e,s) con 6+3 = 9 → fusión es
Luego: es+t → est (9), est+_ → est_ (9), l+o → lo (7), lo+w → low (7)
Palabra nueva "lowest" → low + est_   (sin <UNK>)
```


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("llm", seed=64)
show(result)


## Reflexión

1. El laboratorio `llm` es un modelo didáctico determinista. En un LLM real, ¿por qué el
   mismo prompt puede costar distinto número de tokens en GPT y en BERT, y qué implica
   para comparar "longitudes de contexto" entre modelos?
2. Si tu producto atiende usuarios en español y guaraní con un tokenizador entrenado sobre
   todo en inglés, ¿quién paga más por mensaje y qué medirías para cuantificarlo?
3. ¿Qué se rompe aguas abajo (índices, cachés, embeddings) si actualizas el tokenizador de
   un sistema en producción sin reentrenar el modelo?
